首先，让我们确认三个模型的 backbone 是一样的。

In [2]:
import os
model_path = "/mnt/dolphinfs/hdd_pool/docker/user/hadoop-mtsearch-assistant/ai-search/deepsearch_files/LLMbasemodels/huggingface.co/Qwen"
model_vanilla = os.path.join(model_path, "Qwen3-30B-A3B")
model_w_instruct = os.path.join(model_path, "Qwen3-30B-A3B-Instruct-2507")
model_w_thinking = os.path.join(model_path, "Qwen3-30B-A3B-Thinking-2507")

In [6]:
import json
from safetensors.torch import load_file

In [13]:
model_path = model_vanilla
index_path = os.path.join(model_path, "model.safetensors.index.json")
with open(index_path, "r") as f:
    index = json.load(f)
index["metadata"]["total_size"], index["weight_map"]["lm_head.weight"], len(index["weight_map"])

(61064245248, 'model-00016-of-00016.safetensors', 18867)

In [22]:
weight_map = index["weight_map"]
shared_files = set(weight_map.values())
len(shared_files)

16

In [25]:
state_dict = {}
for file in shared_files:
    shard_path = os.path.join(model_path, file)
    shard_dict = load_file(shard_path)
    state_dict.update(shard_dict)
{ k: state_dict[k] for k in list(state_dict.keys())[:3] }

{'lm_head.weight': tensor([[-0.0271,  0.0422, -0.0295,  ...,  0.0044, -0.0227,  0.0025],
         [ 0.0447,  0.0190,  0.0457,  ...,  0.0282, -0.0374,  0.0032],
         [ 0.0200,  0.0020,  0.0091,  ...,  0.0064, -0.0361,  0.0347],
         ...,
         [-0.0032,  0.0010,  0.0040,  ..., -0.0130,  0.0228, -0.0145],
         [ 0.0044, -0.0019,  0.0019,  ..., -0.0104,  0.0104, -0.0154],
         [ 0.0045, -0.0055, -0.0056,  ..., -0.0128,  0.0226, -0.0154]],
        dtype=torch.bfloat16),
 'model.layers.47.input_layernorm.weight': tensor([0.9844, 1.1562, 0.6133,  ..., 0.5977, 0.6406, 0.8477],
        dtype=torch.bfloat16),
 'model.layers.47.mlp.experts.100.down_proj.weight': tensor([[-0.0112, -0.0165, -0.0010,  ..., -0.0022, -0.0272,  0.0303],
         [ 0.0089, -0.0244,  0.0098,  ..., -0.0043,  0.0092, -0.0146],
         [ 0.0201,  0.0042, -0.0094,  ..., -0.0226,  0.0100,  0.0043],
         ...,
         [-0.0210,  0.0732, -0.0042,  ...,  0.0031,  0.0147, -0.0021],
         [-0.0302, -0.0

In [ ]:
from collections import OrderedDict
backbone_params = OrderedDict()
for k, v in state_dict.items():
    if "lm_head" in k or "score" in k or "classifier" in k or "qa_outputs" in k:
        print(k, v.shape)
    backbone_params[k] = v

lm_head.weight torch.Size([151936, 2048])


封装如上两个函数。

In [ ]:
def load_model_state_dict(model_path):
    index_path = os.path.join(model_path, "model.safetensors.index.json")
    with open(index_path, "r") as f:
        index = json.load(f)
    weight_map = index["weight_map"]
    shared_files = set(weight_map.values())
    state_dict = {}
    for file in shared_files:
        shard_path = os.path.join(model_path, file)
        shard_dict = load_file(shard_path)
        state_dict.update(shard_dict)
    return state_dict

def extract_backbone_params(state_dict):
    backbone_params = OrderedDict()
    for k, v in state_dict.items():
        if "lm_head" in k or "score" in k or "classifier" in k or "qa_outputs" in k:
            print(k, v.shape)
        backbone_params[k] = v
    return backbone_params

首先，经过实证，`lm_head.weight` 的权重都是同形状的，这也自然，即预训练和后训练阶段的 token set 保持一致。

In [36]:
model_paths = [model_vanilla, model_w_instruct, model_w_thinking]
model_names = ["Vanilla", "Instruct", "Thinking"]
state_dicts = {}
backbone_dicts = {}

for model_path, model_name in zip(model_paths, model_names):
    print(f"\nLoading {model_name}...")
    state_dict = load_model_state_dict(model_path)
    state_dicts[model_name] = state_dict
    backbone_dicts[model_name] = extract_backbone_params(state_dict)
    print(f"  Total parameters: {len(state_dict)}")
    print(f"  Backbone parameters: {len(backbone_dicts[model_name])}")


Loading Vanilla...
lm_head.weight torch.Size([151936, 2048])
  Total parameters: 18867
  Backbone parameters: 18867

Loading Instruct...
lm_head.weight torch.Size([151936, 2048])
  Total parameters: 18867
  Backbone parameters: 18867

Loading Thinking...
lm_head.weight torch.Size([151936, 2048])
  Total parameters: 18867
  Backbone parameters: 18867


In [37]:
ref_name = model_names[0]
ref_backbone = backbone_dicts[ref_name]

这里首先比较参数的 keys，即模型的层和模块命名是否相同。结果是完全相同。

In [ ]:
for model_name in model_names[1:]:
    ref_keys = set(ref_backbone.keys())
    model_keys = set(backbone_dicts[model_name].keys())
    if ref_keys == model_keys:
        print(f"{model_name}: All backbone keys match ({len(ref_keys)} keys)")
    else:
        missing_in_model = ref_keys - model_keys
        extra_in_model = model_keys - ref_keys
        if missing_in_model:
            print(f"{model_name}: Missing keys ({len(missing_in_model)}): {list(missing_in_model)[:5]}...")
        if extra_in_model:
            print(f"{model_name}: Extra keys ({len(extra_in_model)}): {list(extra_in_model)[:5]}...")

Instruct: All backbone keys match (18867 keys)
Thinking: All backbone keys match (18867 keys)


下一步比较不同参数的 shape。结果也是都匹配。

In [ ]:
for model_name in model_names[1:]:
    mismatched_shapes = []
    for key in ref_backbone.keys():
        if key not in backbone_dicts[model_name]:
            print(f"{model_name}: Missing key: {key}")
            continue
        ref_shape = ref_backbone[key].shape
        model_shape = backbone_dicts[model_name][key].shape
        
        if ref_shape != model_shape:
            print(f"{model_name}: Shape mismatch for key: {key}")
    print(f"{model_name}: All shapes match")